# Notebook 09: Automated Testing

## Purpose

This notebook creates automated tests for the reusable PaySim pipeline modules.

The tests use a small synthetic transaction dataset rather than the complete
6.3-million-row PaySim file.

## Modules tested

- `bronze.py`
- `silver.py`
- `gold.py`
- `validation.py`
- `audit.py`
- `io_utils.py`

## Testing goals

1. Verify raw-to-canonical column standardization.
2. Verify Bronze metadata generation.
3. Verify Silver validity rules.
4. Verify time and balance features.
5. Verify high-value indicators.
6. Verify Gold table grains and aggregations.
7. Verify fraud-feature history uses prior records.
8. Verify validation failures are detected.
9. Verify audit records are generated.
10. Verify small CSV exports.

In [2]:
# %pip install pytest pytest-cov

In [3]:
from pathlib import Path
import os
import subprocess
import sys

In [4]:
current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

SRC_PATH = PROJECT_ROOT / "src"
TESTS_PATH = PROJECT_ROOT / "tests"

TESTS_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)
print("Tests path:", TESTS_PATH)

Project root: C:\Projects\paysim-financial-data-pipeline
Source path: C:\Projects\paysim-financial-data-pipeline\src
Tests path: C:\Projects\paysim-financial-data-pipeline\tests


In [5]:
PACKAGE_PATH = (
    SRC_PATH
    / "paysim_pipeline"
)

expected_modules = {
    "__init__.py",
    "audit.py",
    "bronze.py",
    "config.py",
    "gold.py",
    "io_utils.py",
    "schemas.py",
    "silver.py",
    "spark_session.py",
    "validation.py",
}

actual_modules = {
    path.name
    for path in PACKAGE_PATH.glob("*.py")
}

missing_modules = (
    expected_modules - actual_modules
)

print("Missing modules:", missing_modules)

assert not missing_modules

print("Pipeline-module validation: PASS")

Missing modules: set()
Pipeline-module validation: PASS


In [6]:
%%writefile ../pytest.ini
[pytest]
testpaths = tests
python_files = test_*.py
python_functions = test_*
addopts = -ra

Writing ../pytest.ini


In [7]:
%%writefile ../tests/conftest.py
"""Shared PySpark fixtures for pipeline tests."""

import os
import sys
from pathlib import Path

import pytest

from pyspark.sql import SparkSession
from pyspark.sql import types as T


PROJECT_ROOT = Path(
    __file__
).resolve().parents[1]

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )


@pytest.fixture(scope="session")
def spark():
    """Create one Spark session for the test run."""

    python_executable = sys.executable

    os.environ["PYSPARK_PYTHON"] = (
        python_executable
    )
    os.environ["PYSPARK_DRIVER_PYTHON"] = (
        python_executable
    )

    session = (
        SparkSession.builder
        .appName("PaySimPipelineTests")
        .master("local[2]")
        .config(
            "spark.driver.memory",
            "2g",
        )
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.session.timeZone",
            "UTC",
        )
        .config(
            "spark.driver.host",
            "127.0.0.1",
        )
        .config(
            "spark.driver.bindAddress",
            "127.0.0.1",
        )
        .getOrCreate()
    )

    session.sparkContext.setLogLevel(
        "ERROR"
    )

    yield session

    session.catalog.clearCache()
    session.stop()


@pytest.fixture()
def bronze_schema():
    """Canonical Bronze schema for tests."""

    return T.StructType(
        [
            T.StructField(
                "step",
                T.IntegerType(),
                False,
            ),
            T.StructField(
                "transaction_type",
                T.StringType(),
                False,
            ),
            T.StructField(
                "amount",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "origin_account",
                T.StringType(),
                False,
            ),
            T.StructField(
                "origin_old_balance",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "origin_new_balance",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "destination_account",
                T.StringType(),
                False,
            ),
            T.StructField(
                "destination_old_balance",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "destination_new_balance",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "is_fraud",
                T.IntegerType(),
                False,
            ),
            T.StructField(
                "is_flagged_fraud",
                T.IntegerType(),
                False,
            ),
            T.StructField(
                "pipeline_run_id",
                T.StringType(),
                False,
            ),
        ]
    )


@pytest.fixture()
def bronze_df(
    spark,
    bronze_schema,
):
    """Create a small canonical Bronze dataset."""

    rows = [
        (
            1,
            "PAYMENT",
            100.0,
            "C001",
            1000.0,
            900.0,
            "M001",
            0.0,
            0.0,
            0,
            0,
            "test_run_001",
        ),
        (
            2,
            "TRANSFER",
            300000.0,
            "C002",
            500000.0,
            200000.0,
            "C003",
            0.0,
            300000.0,
            1,
            1,
            "test_run_001",
        ),
        (
            25,
            "CASH_OUT",
            500.0,
            "C002",
            200000.0,
            199500.0,
            "C004",
            0.0,
            500.0,
            1,
            0,
            "test_run_001",
        ),
        (
            26,
            "CASH_IN",
            0.0,
            "C005",
            1000.0,
            1000.0,
            "C006",
            100.0,
            100.0,
            0,
            0,
            "test_run_001",
        ),
    ]

    return spark.createDataFrame(
        rows,
        schema=bronze_schema,
    )


@pytest.fixture()
def silver_df(bronze_df):
    """Create Silver transactions for tests."""

    from paysim_pipeline.silver import (
        build_silver_transactions,
    )

    return build_silver_transactions(
        bronze_dataframe=bronze_df,
        high_value_threshold=200000.0,
    )

Writing ../tests/conftest.py


In [8]:
%%writefile ../tests/test_bronze.py
"""Tests for Bronze-layer functions."""

from pyspark.sql import types as T

from paysim_pipeline.bronze import (
    add_bronze_metadata,
    standardize_bronze_columns,
)


def test_standardize_bronze_columns(
    spark,
):
    """Raw columns should receive canonical names."""

    raw_schema = T.StructType(
        [
            T.StructField(
                "step",
                T.IntegerType(),
                False,
            ),
            T.StructField(
                "type",
                T.StringType(),
                False,
            ),
            T.StructField(
                "amount",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "nameOrig",
                T.StringType(),
                False,
            ),
            T.StructField(
                "oldbalanceOrg",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "newbalanceOrig",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "nameDest",
                T.StringType(),
                False,
            ),
            T.StructField(
                "oldbalanceDest",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "newbalanceDest",
                T.DoubleType(),
                False,
            ),
            T.StructField(
                "isFraud",
                T.IntegerType(),
                False,
            ),
            T.StructField(
                "isFlaggedFraud",
                T.IntegerType(),
                False,
            ),
        ]
    )

    raw_df = spark.createDataFrame(
        [
            (
                1,
                "PAYMENT",
                100.0,
                "C001",
                1000.0,
                900.0,
                "M001",
                0.0,
                0.0,
                0,
                0,
            )
        ],
        schema=raw_schema,
    )

    result_df = standardize_bronze_columns(
        raw_dataframe=raw_df
    )

    assert "transaction_type" in (
        result_df.columns
    )
    assert "origin_account" in (
        result_df.columns
    )
    assert "destination_account" in (
        result_df.columns
    )
    assert "is_fraud" in result_df.columns
    assert "type" not in result_df.columns
    assert "nameOrig" not in result_df.columns


def test_add_bronze_metadata(
    spark,
):
    """Bronze records should contain audit metadata."""

    source_df = spark.createDataFrame(
        [(1,)],
        ["step"],
    )

    result_df = add_bronze_metadata(
        dataframe=source_df,
        source_file_name="test.csv",
        pipeline_run_id="run_001",
    )

    row = result_df.first()

    assert row.source_file_name == "test.csv"
    assert row.pipeline_run_id == "run_001"
    assert (
        row.bronze_ingestion_timestamp
        is not None
    )

Writing ../tests/test_bronze.py


In [9]:
%%writefile ../tests/test_silver.py
"""Tests for Silver-layer transformations."""

from paysim_pipeline.silver import (
    build_silver_transactions,
)


def test_silver_preserves_valid_rows(
    bronze_df,
):
    """All valid test transactions should remain."""

    silver_df = build_silver_transactions(
        bronze_dataframe=bronze_df,
        high_value_threshold=200000.0,
    )

    assert silver_df.count() == 4


def test_negative_amount_is_rejected(
    spark,
    bronze_df,
):
    """Transactions with negative amounts are invalid."""

    invalid_row = (
        3,
        "PAYMENT",
        -10.0,
        "C010",
        100.0,
        110.0,
        "M010",
        0.0,
        0.0,
        0,
        0,
        "test_run_001",
    )

    invalid_df = spark.createDataFrame(
        [invalid_row],
        schema=bronze_df.schema,
    )

    combined_df = bronze_df.unionByName(
        invalid_df
    )

    silver_df = build_silver_transactions(
        bronze_dataframe=combined_df,
        high_value_threshold=200000.0,
    )

    assert combined_df.count() == 5
    assert silver_df.count() == 4


def test_time_features(silver_df):
    """Steps should map to correct day and hour."""

    time_rows = {
        row.step: (
            row.transaction_day,
            row.transaction_hour,
        )
        for row in (
            silver_df
            .select(
                "step",
                "transaction_day",
                "transaction_hour",
            )
            .collect()
        )
    }

    assert time_rows[1] == (1, 0)
    assert time_rows[2] == (1, 1)
    assert time_rows[25] == (2, 0)
    assert time_rows[26] == (2, 1)


def test_high_value_indicator(silver_df):
    """Only the 300,000 transaction is high value."""

    high_value_rows = (
        silver_df
        .filter(
            "is_high_value_transaction = 1"
        )
        .select("amount")
        .collect()
    )

    assert len(high_value_rows) == 1
    assert high_value_rows[0].amount == 300000.0


def test_origin_balance_error(silver_df):
    """Balanced origin transactions have zero error."""

    errors = [
        row.origin_balance_error
        for row in (
            silver_df
            .select(
                "origin_balance_error"
            )
            .collect()
        )
    ]

    assert all(
        error == 0.0
        for error in errors
    )


def test_destination_merchant_indicator(
    silver_df,
):
    """M-prefixed destinations are merchants."""

    payment_row = (
        silver_df
        .filter("step = 1")
        .select(
            "is_destination_merchant"
        )
        .first()
    )

    assert (
        payment_row
        .is_destination_merchant
        == 1
    )

Writing ../tests/test_silver.py


In [10]:
%%writefile ../tests/test_gold.py
"""Tests for Gold analytical tables."""

from paysim_pipeline.gold import (
    build_gold_tables,
    create_daily_transaction_summary,
    create_fraud_feature_table,
    create_fraud_monitoring_table,
    create_high_value_summary,
    create_transaction_type_summary,
)


def test_daily_summary_grain(silver_df):
    """Daily summary should have one row per day."""

    result_df = (
        create_daily_transaction_summary(
            silver_dataframe=silver_df,
            approximate_distinct_rsd=0.05,
        )
    )

    assert result_df.count() == 2

    days = {
        row.transaction_day
        for row in (
            result_df
            .select("transaction_day")
            .collect()
        )
    }

    assert days == {1, 2}


def test_daily_summary_reconciles(
    silver_df,
):
    """Daily transaction counts should reconcile."""

    result_df = (
        create_daily_transaction_summary(
            silver_dataframe=silver_df,
            approximate_distinct_rsd=0.05,
        )
    )

    reconciled_count = (
        result_df
        .groupBy()
        .sum("transaction_count")
        .first()[0]
    )

    assert reconciled_count == 4


def test_daily_fraud_counts(silver_df):
    """Daily fraud counts should equal Silver."""

    result_df = (
        create_daily_transaction_summary(
            silver_dataframe=silver_df,
            approximate_distinct_rsd=0.05,
        )
    )

    daily_fraud_count = (
        result_df
        .groupBy()
        .sum("fraud_count")
        .first()[0]
    )

    assert daily_fraud_count == 2


def test_transaction_type_summary(
    silver_df,
):
    """Each transaction type should appear once."""

    result_df = (
        create_transaction_type_summary(
            silver_dataframe=silver_df
        )
    )

    assert result_df.count() == 4

    reconciled_count = (
        result_df
        .groupBy()
        .sum("transaction_count")
        .first()[0]
    )

    assert reconciled_count == 4


def test_high_value_summary(silver_df):
    """Only one test transaction is high value."""

    result_df = create_high_value_summary(
        silver_dataframe=silver_df
    )

    assert result_df.count() == 1

    row = result_df.first()

    assert (
        row.high_value_transaction_count
        == 1
    )
    assert (
        row.high_value_total_amount
        == 300000.0
    )
    assert row.high_value_fraud_count == 1


def test_fraud_monitoring_table(
    silver_df,
):
    """Fraud monitoring contains fraud only."""

    result_df = (
        create_fraud_monitoring_table(
            silver_dataframe=silver_df
        )
    )

    assert result_df.count() == 2

    fraud_values = {
        row.is_fraud
        for row in (
            result_df
            .select("is_fraud")
            .collect()
        )
    }

    assert fraud_values == {1}


def test_fraud_features_use_prior_records(
    silver_df,
):
    """C002's second transaction sees one prior record."""

    feature_df = (
        create_fraud_feature_table(
            silver_dataframe=silver_df
        )
    )

    row = (
        feature_df
        .filter(
            "origin_account = 'C002' "
            "AND step = 25"
        )
        .first()
    )

    assert row is not None
    assert (
        row.prior_origin_transaction_count
        == 1
    )
    assert (
        row.prior_origin_average_amount
        == 300000.0
    )


def test_build_gold_table_registry(
    silver_df,
):
    """Gold builder should return all expected tables."""

    gold_tables = build_gold_tables(
        silver_dataframe=silver_df,
        approximate_distinct_rsd=0.05,
    )

    expected_tables = {
        "daily_transaction_summary",
        "daily_type_summary",
        "hourly_fraud_summary",
        "transaction_type_summary",
        "origin_account_summary",
        "destination_account_summary",
        "high_value_summary",
        "fraud_monitoring",
        "fraud_feature",
    }

    assert set(gold_tables) == expected_tables

Writing ../tests/test_gold.py


In [11]:
%%writefile ../tests/test_validation.py
"""Tests for data-quality validation functions."""

import pytest

from paysim_pipeline.validation import (
    build_null_profile,
    count_duplicate_transactions,
    find_missing_columns,
    reconciliation_result,
    validate_binary_column,
    validate_required_columns,
)


def test_find_missing_columns(spark):
    """Missing columns should be returned."""

    dataframe = spark.createDataFrame(
        [(1, "PAYMENT")],
        ["step", "transaction_type"],
    )

    missing_columns = find_missing_columns(
        dataframe=dataframe,
        required_columns=[
            "step",
            "transaction_type",
            "amount",
        ],
    )

    assert missing_columns == ["amount"]


def test_validate_required_columns_passes(
    spark,
):
    """Validation should pass when columns exist."""

    dataframe = spark.createDataFrame(
        [(1, 100.0)],
        ["step", "amount"],
    )

    validate_required_columns(
        dataframe=dataframe,
        required_columns=[
            "step",
            "amount",
        ],
        dataframe_name="test_dataframe",
    )


def test_validate_required_columns_fails(
    spark,
):
    """Validation should fail when a column is absent."""

    dataframe = spark.createDataFrame(
        [(1,)],
        ["step"],
    )

    with pytest.raises(
        ValueError,
        match="missing required columns",
    ):
        validate_required_columns(
            dataframe=dataframe,
            required_columns=[
                "step",
                "amount",
            ],
            dataframe_name="test_dataframe",
        )


def test_null_profile(spark):
    """Null profile should count null values."""

    dataframe = spark.createDataFrame(
        [
            (1, "A"),
            (2, None),
            (None, "C"),
        ],
        ["number", "letter"],
    )

    result = build_null_profile(
        dataframe
    ).first()

    assert result.number == 1
    assert result.letter == 1


def test_duplicate_count(spark):
    """Duplicate business keys should be counted."""

    dataframe = spark.createDataFrame(
        [
            (1, "A"),
            (1, "A"),
            (2, "B"),
        ],
        ["step", "account"],
    )

    duplicate_group_count = (
        count_duplicate_transactions(
            dataframe=dataframe,
            key_columns=[
                "step",
                "account",
            ],
        )
    )

    assert duplicate_group_count == 1


def test_binary_column_validation(spark):
    """Invalid binary values should be returned."""

    dataframe = spark.createDataFrame(
        [
            (0,),
            (1,),
            (2,),
            (None,),
        ],
        ["indicator"],
    )

    invalid_values = {
        row.indicator
        for row in (
            validate_binary_column(
                dataframe=dataframe,
                column_name="indicator",
            )
            .collect()
        )
    }

    assert invalid_values == {2}


def test_successful_reconciliation():
    """Equal counts should produce PASS."""

    result = reconciliation_result(
        source_count=100,
        target_count=100,
    )

    assert (
        result["reconciliation_status"]
        == "PASS"
    )
    assert result["row_count_difference"] == 0


def test_failed_reconciliation():
    """Different counts should produce FAIL."""

    result = reconciliation_result(
        source_count=100,
        target_count=95,
    )

    assert (
        result["reconciliation_status"]
        == "FAIL"
    )
    assert (
        result["row_count_difference"]
        == -5
    )

Writing ../tests/test_validation.py


In [12]:
%%writefile ../tests/test_audit_io.py
"""Tests for audit and output utilities."""

import pandas as pd

from paysim_pipeline.audit import (
    combine_audit_records,
    create_audit_record,
)

from paysim_pipeline.io_utils import (
    clear_csv_outputs,
    export_small_dataframe_to_csv,
)


def test_create_audit_record(spark):
    """Audit record should describe its DataFrame."""

    dataframe = spark.createDataFrame(
        [
            (1, "A"),
            (2, "B"),
        ],
        ["id", "value"],
    )

    audit_df = create_audit_record(
        spark=spark,
        dataframe=dataframe,
        pipeline_run_id="run_001",
        pipeline_stage="TEST",
        table_name="test_table",
        execution_time_seconds=1.5,
        reconciliation_status="PASS",
    )

    row = audit_df.first()

    assert row.pipeline_run_id == "run_001"
    assert row.pipeline_stage == "TEST"
    assert row.table_name == "test_table"
    assert row.row_count == 2
    assert row.column_count == 2
    assert row.reconciliation_status == "PASS"


def test_combine_audit_records(spark):
    """Multiple audit records should union."""

    first_df = create_audit_record(
        spark=spark,
        dataframe=spark.range(2),
        pipeline_run_id="run_001",
        pipeline_stage="BRONZE",
        table_name="bronze_test",
        execution_time_seconds=1.0,
    )

    second_df = create_audit_record(
        spark=spark,
        dataframe=spark.range(3),
        pipeline_run_id="run_001",
        pipeline_stage="SILVER",
        table_name="silver_test",
        execution_time_seconds=2.0,
    )

    combined_df = combine_audit_records(
        [first_df, second_df]
    )

    assert combined_df.count() == 2


def test_export_small_dataframe_to_csv(
    spark,
    tmp_path,
):
    """Small Spark tables should export to CSV."""

    dataframe = spark.createDataFrame(
        [
            (1, "A"),
            (2, "B"),
        ],
        ["id", "value"],
    )

    output_path = (
        export_small_dataframe_to_csv(
            dataframe=dataframe,
            output_path=tmp_path,
            file_name="test_output.csv",
        )
    )

    assert output_path.exists()

    exported_df = pd.read_csv(
        output_path
    )

    assert len(exported_df) == 2
    assert list(exported_df.columns) == [
        "id",
        "value",
    ]


def test_clear_csv_outputs(tmp_path):
    """CSV cleanup should preserve other files."""

    first_csv = tmp_path / "first.csv"
    second_csv = tmp_path / "second.csv"
    text_file = tmp_path / "notes.txt"

    first_csv.write_text(
        "id\n1\n",
        encoding="utf-8",
    )

    second_csv.write_text(
        "id\n2\n",
        encoding="utf-8",
    )

    text_file.write_text(
        "retain me",
        encoding="utf-8",
    )

    deleted_files = clear_csv_outputs(
        output_path=tmp_path
    )

    assert len(deleted_files) == 2
    assert not first_csv.exists()
    assert not second_csv.exists()
    assert text_file.exists()

Writing ../tests/test_audit_io.py


In [13]:
sorted(
    path.name
    for path in TESTS_PATH.glob(
        "test_*.py"
    )
)

['test_audit_io.py',
 'test_bronze.py',
 'test_gold.py',
 'test_silver.py',
 'test_validation.py']

In [14]:
compile_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "compileall",
        "-q",
        str(TESTS_PATH),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(
    "Compile return code:",
    compile_result.returncode,
)

if compile_result.stdout:
    print(compile_result.stdout)

if compile_result.stderr:
    print(compile_result.stderr)

assert compile_result.returncode == 0

print("Test-file syntax validation: PASS")

Compile return code: 0
Test-file syntax validation: PASS


In [15]:
test_environment = os.environ.copy()

test_environment["PYTHONPATH"] = str(
    SRC_PATH
)

test_environment["PYSPARK_PYTHON"] = (
    sys.executable
)

test_environment[
    "PYSPARK_DRIVER_PYTHON"
] = sys.executable

print("Test environment prepared.")

Test environment prepared.


In [16]:
test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-v",
    ],
    cwd=PROJECT_ROOT,
    env=test_environment,
    capture_output=True,
    text=True,
)

print(test_result.stdout)

if test_result.stderr:
    print(test_result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.12.5, pytest-9.1.1, pluggy-1.6.0 -- c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Projects\paysim-financial-data-pipeline
configfile: pytest.ini
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 28 items

tests/test_audit_io.py::test_create_audit_record PASSED                  [  3%]
tests/test_audit_io.py::test_combine_audit_records PASSED                [  7%]
tests/test_audit_io.py::test_export_small_dataframe_to_csv PASSED        [ 10%]
tests/test_audit_io.py::test_clear_csv_outputs PASSED                    [ 14%]
tests/test_bronze.py::test_standardize_bronze_columns PASSED             [ 17%]
tests/test_bronze.py::test_add_bronze_metadata PASSED                    [ 21%]
tests/test_gold.py::test_daily_summary_grain PASSED                      [ 25%]
tests/test_gold.py::test_daily_summary_reconciles PASSED      

In [17]:
print(
    "Pytest return code:",
    test_result.returncode,
)

assert test_result.returncode == 0, (
    "One or more automated tests failed. "
    "Review the pytest output above."
)

print("Complete automated test suite: PASS")

Pytest return code: 0
Complete automated test suite: PASS


In [18]:
coverage_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "--cov=paysim_pipeline",
        "--cov-report=term-missing",
        "-q",
    ],
    cwd=PROJECT_ROOT,
    env=test_environment,
    capture_output=True,
    text=True,
)

print(coverage_result.stdout)

if coverage_result.stderr:
    print(coverage_result.stderr)

............................                                             [100%]
============================== warnings summary ===============================
tests/test_audit_io.py::test_create_audit_record
  c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
    require_minimum_pandas_version()

tests/test_audit_io.py::test_export_small_dataframe_to_csv
  c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
    require_minimum_pandas_version()

tests/test_audit_io.py::test_export_small_dataframe_to_csv
  c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pa

In [19]:
print(
    "Coverage return code:",
    coverage_result.returncode,
)

assert coverage_result.returncode == 0

print("Coverage execution: PASS")

Coverage return code: 0
Coverage execution: PASS


## Test architecture

### Unit tests

Test individual functions using small deterministic DataFrames:

- column renaming;
- feature derivation;
- validation rules;
- aggregation logic;
- audit creation.

### Integration tests

Test multiple functions together:

- Bronze fixture to Silver transformation;
- Silver fixture to Gold tables;
- Spark DataFrame to CSV output.

### Full pipeline test

A future integration test can execute the complete pipeline against a small
temporary CSV file.

The full 6.3-million-row source dataset is not required for routine unit tests.

## Regression protection

The automated tests protect the pipeline from changes such as:

1. accidentally removing required columns;
2. changing transaction-day calculations;
3. including negative transaction amounts;
4. changing the high-value threshold logic;
5. breaking Gold-table reconciliation;
6. including non-fraud records in fraud monitoring;
7. using future records in historical fraud features;
8. generating incorrect audit row counts;
9. breaking CSV export behavior.

If a future code change violates one of these expectations, pytest will report
the affected test before the pipeline is released.

# Notebook 09 completion summary

Automated tests were created for the reusable PaySim pipeline.

## Test coverage

- Bronze standardization
- Bronze metadata
- Silver filtering
- Time-feature creation
- Balance validation
- High-value classification
- Gold aggregation grains
- Gold reconciliation
- Fraud monitoring
- Prior-account fraud features
- Required-column validation
- Null profiling
- Duplicate detection
- Binary-indicator validation
- Audit generation
- CSV output and cleanup

## Engineering outcome

The project is no longer validated only through manual notebook inspection.

Pipeline behavior is now documented as executable tests that can be run after
every code change with:

`python -m pytest tests -v`

## Next project step

Create a command-line entry point:

`src/paysim_pipeline/main.py`

This will allow the modular pipeline to run outside Jupyter and prepare it for
scheduling, Docker, and CI/CD.